# 11 — Junk Dimension — DuckDB

Dimensão que agrupa flags de baixa cardinalidade do FactSales.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())
from utils import get_conn, DB_PATH, DATA_DIR

conn = get_conn()
print(f"Conectado: {DB_PATH}")

Conectado: /workspace/pf_northwind/duckdb/northwind_dw.duckdb


In [2]:
# ============================================================
# Passo 1: Criar DimOrderFlags com 24 combinações (4x2x3)
# ============================================================
conn.execute("""
    CREATE OR REPLACE TABLE gold.DimOrderFlags AS
    SELECT
        ROW_NUMBER() OVER () AS OrderFlagsSK,
        d.DiscountBand,
        v.IsHighValue,
        s.ShipmentMode
    FROM (VALUES ('None'), ('Low'), ('Medium'), ('High')) AS d(DiscountBand)
    CROSS JOIN (VALUES (false), (true)) AS v(IsHighValue)
    CROSS JOIN (VALUES ('Express'), ('Standard'), ('Economy')) AS s(ShipmentMode)
""")

n = conn.execute("SELECT COUNT(*) AS n FROM gold.DimOrderFlags").fetchdf()['n'][0]
assert n == 24, f"Esperado 24, got {n}"
print(f"✓ DimOrderFlags: {n} combinações")


✓ DimOrderFlags: 24 combinações


In [3]:
# ============================================================
# Passo 2: Adicionar coluna e popular FactSales
# ============================================================
conn.execute("ALTER TABLE gold.FactSales ADD COLUMN IF NOT EXISTS OrderFlagsSK INTEGER")

conn.execute("""
    UPDATE gold.FactSales fs
    SET OrderFlagsSK = (
        SELECT dof.OrderFlagsSK
        FROM gold.DimOrderFlags dof
        JOIN bronze.Orders o ON o.OrderID = fs.OrderID
        WHERE dof.DiscountBand = CASE
              WHEN fs.Discount = 0     THEN 'None'
              WHEN fs.Discount <= 0.05 THEN 'Low'
              WHEN fs.Discount <= 0.15 THEN 'Medium'
              ELSE 'High' END
          AND dof.IsHighValue = CASE WHEN fs.NetRevenue > 1000 THEN true ELSE false END
          AND dof.ShipmentMode = CASE o.ShipVia
              WHEN 1 THEN 'Express'
              WHEN 2 THEN 'Standard'
              ELSE 'Economy' END
        LIMIT 1
    )
""")

null_count = conn.execute("SELECT COUNT(*) AS n FROM gold.FactSales WHERE OrderFlagsSK IS NULL").fetchdf()['n'][0]
assert null_count == 0, f"OrderFlagsSK com NULL: {null_count}"
print(f"✓ FactSales atualizado, sem NULLs em OrderFlagsSK")


✓ FactSales atualizado, sem NULLs em OrderFlagsSK


In [4]:
# ============================================================
# DEMO: Distribuição por DiscountBand e ShipmentMode
# ============================================================
conn.execute("""
    SELECT dof.DiscountBand, dof.ShipmentMode,
           COUNT(*) AS Transacoes,
           ROUND(SUM(fs.GrossRevenue), 2) AS GrossRevenue,
           ROUND(SUM(fs.NetRevenue), 2) AS NetRevenue
    FROM gold.FactSales fs
    JOIN gold.DimOrderFlags dof ON dof.OrderFlagsSK = fs.OrderFlagsSK
    GROUP BY dof.DiscountBand, dof.ShipmentMode
    ORDER BY dof.DiscountBand, dof.ShipmentMode
""").fetchdf()


,DiscountBand,ShipmentMode,Transacoes,GrossRevenue,NetRevenue
0,High,Economy,140,98097.05,78820.84
1,High,Express,152,98566.09,79484.97
2,High,Standard,180,149679.77,117320.03
3,Low,Standard,7,296.65,288.15
4,Medium,Economy,89,69163.08,64093.94
5,Medium,Express,132,91067.83,85005.70
6,Medium,Standard,138,96889.51,90080.80
7,None,Economy,416,240490.69,240490.69
8,None,Express,362,184349.27,184349.27
9,None,Standard,539,325858.65,325858.65
